# 08 — Real-World NLP Pipeline
**Goal:** Construir un sistema completo de análisis de voz del cliente para un equipo de growth.

Simularemos tener una encuesta NPS con comentarios abiertos y construiremos:
1. Clasificación automática de temas
2. Análisis de sentimiento por canal de adquisición
3. Detección de tickets urgentes
4. Extracción de señales para el funnel
5. Dashboard de resumen textual

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})

STOPWORDS_ES = [
    'a','al','algo','ante','como','con','de','del','desde','el','ella','en',
    'entre','es','esta','este','esto','fue','la','las','le','lo','los','me',
    'mi','muy','no','nos','o','para','pero','por','que','se','si','sin',
    'su','sus','también','te','todo','un','una','y','ya','yo','tuve','llevo',
]

def preprocess(text):
    text = text.lower()
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'[^\wáéíóúüñ\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

print('Setup completo.')

## 1. Generar dataset de encuesta NPS simulada

In [ ]:
rng = np.random.default_rng(42)

CHANNELS = ['organic', 'paid', 'email', 'social', 'direct']

# Comentarios por categoría con variaciones de sentimiento
COMMENTS = {
    # (text, category, sentiment)
    'app_ux': [
        ('La app está muy lenta y tarda en cargar', 0),
        ('La aplicación no carga, da error al abrir', 0),
        ('La app mejoró mucho, ahora carga rápido', 1),
        ('La interfaz es confusa y difícil de usar', 0),
        ('App muy intuitiva, fácil de navegar', 1),
        ('Se cierra sola cuando intento iniciar sesión', 0),
    ],
    'otp_seguridad': [
        ('No me llegó el OTP al celular', 0),
        ('El código OTP no llega por SMS', 0),
        ('El OTP llegó rápido y sin problemas', 1),
        ('El sistema de verificación es seguro y fácil', 1),
        ('No recibo el SMS de verificación', 0),
    ],
    'documentos': [
        ('La carga de documentos falla constantemente', 0),
        ('No puedo subir mis documentos al sistema', 0),
        ('Subir los documentos fue muy sencillo', 1),
        ('El sistema no acepta mi documento de identidad', 0),
    ],
    'credito': [
        ('Llevo días esperando la evaluación crediticia', 0),
        ('Aprobaron mi crédito en minutos, increíble', 1),
        ('Por qué rechazaron mi solicitud sin explicación', 0),
        ('El proceso de aprobación fue muy rápido', 1),
        ('No hay información sobre mi evaluación', 0),
    ],
    'soporte': [
        ('El call center no responde, pésimo servicio', 0),
        ('Nadie del soporte resuelve mis problemas', 0),
        ('La atención al cliente fue excelente', 1),
        ('El agente resolvió todo rápido y amable', 1),
        ('Llamé cuatro veces sin solución', 0),
    ],
    'beneficios': [
        ('Los beneficios de la tarjeta son increíbles', 1),
        ('Muy buenos descuentos y cashback', 1),
        ('Los beneficios prometidos no existen', 0),
        ('Excelentes promociones, totalmente recomendada', 1),
        ('El cashback es bajo comparado con otras tarjetas', 0),
    ],
}

N = 1000
categories = list(COMMENTS.keys())
rows = []
for _ in range(N):
    cat = categories[rng.integers(len(categories))]
    options = COMMENTS[cat]
    text, sent = options[rng.integers(len(options))]
    channel = CHANNELS[rng.integers(len(CHANNELS))]
    nps = int(rng.integers(0 if sent == 0 else 6, 5 if sent == 0 else 11))
    rows.append({
        'comment': text,
        'category': cat,
        'sentiment': sent,
        'channel': channel,
        'nps_score': nps,
    })

df = pd.DataFrame(rows)
print(f'Encuesta NPS: {len(df)} respuestas')
print(f'NPS score medio: {df.nps_score.mean():.1f}/10')
print(f'% positivos: {df.sentiment.mean():.0%}')
print()
print(df.groupby('category')['sentiment'].agg(['count','mean']).round(2))

## 2. Modelo 1 — Clasificador de categorías

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['comment'], df['category'], test_size=0.2,
    random_state=42, stratify=df['category']
)

cat_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(preprocessor=preprocess, stop_words=STOPWORDS_ES,
                               ngram_range=(1,2), sublinear_tf=True, min_df=2)),
    ('clf', LogisticRegression(max_iter=1000, C=2.0, multi_class='multinomial')),
])
cat_pipe.fit(X_train, y_train)
y_pred_cat = cat_pipe.predict(X_test)

from sklearn.metrics import f1_score
print(f'F1 macro (categorías): {f1_score(y_test, y_pred_cat, average="macro"):.3f}\n')
print(classification_report(y_test, y_pred_cat, digits=3))

## 3. Modelo 2 — Clasificador de sentimiento

In [ ]:
Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    df['comment'], df['sentiment'], test_size=0.2,
    random_state=42, stratify=df['sentiment']
)

sent_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(preprocessor=preprocess, stop_words=STOPWORDS_ES,
                               ngram_range=(1,2), sublinear_tf=True, min_df=2)),
    ('clf', LogisticRegression(max_iter=1000, C=1.0)),
])
sent_pipe.fit(Xs_train, ys_train)
y_pred_sent = sent_pipe.predict(Xs_test)

print(f'F1 (sentimiento): {f1_score(ys_test, y_pred_sent):.3f}')

## 4. Aplicar ambos modelos al dataset completo

In [ ]:
df['pred_category']  = cat_pipe.predict(df['comment'])
df['pred_sentiment'] = sent_pipe.predict(df['comment'])
df['pred_sent_prob'] = sent_pipe.predict_proba(df['comment'])[:, 1]

# Urgency flag: sentimiento negativo + NPS ≤ 4
df['urgent'] = ((df['pred_sentiment'] == 0) & (df['nps_score'] <= 4)).astype(int)

print(f'Tickets urgentes: {df.urgent.sum()} ({df.urgent.mean():.0%})')

## 5. Análisis por canal de adquisición

In [ ]:
channel_summary = (
    df.groupby('channel')
    .agg(
        n=('comment', 'count'),
        nps_mean=('nps_score', 'mean'),
        pct_positivo=('pred_sentiment', 'mean'),
        pct_urgente=('urgent', 'mean'),
        top_issue=('pred_category', lambda x: x.value_counts().index[0])
    )
    .round(3)
    .sort_values('nps_mean', ascending=False)
)
print(channel_summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['#4361ee','#f72585','#06d6a0','#ff9f1c','#7209b7']

chs = channel_summary.index.tolist()

# NPS medio por canal
axes[0].bar(chs, channel_summary['nps_mean'], color=colors)
axes[0].set_title('NPS medio por canal')
axes[0].set_ylabel('NPS score')
axes[0].set_ylim(0, 10)
for i, v in enumerate(channel_summary['nps_mean']):
    axes[0].text(i, v + 0.1, f'{v:.1f}', ha='center', fontsize=9)

# % sentimiento positivo
axes[1].bar(chs, channel_summary['pct_positivo'] * 100, color=colors)
axes[1].set_title('% Comentarios positivos')
axes[1].set_ylabel('%')
axes[1].set_ylim(0, 100)

# % urgentes
axes[2].bar(chs, channel_summary['pct_urgente'] * 100, color='#e63946')
axes[2].set_title('% Tickets urgentes')
axes[2].set_ylabel('%')

for ax in axes:
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('NLP Dashboard — Análisis por canal de adquisición', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Temas dominantes y su relación con NPS

In [ ]:
cat_nps = (
    df.groupby('pred_category')
    .agg(n=('comment','count'), nps_mean=('nps_score','mean'),
         pct_urgente=('urgent','mean'))
    .sort_values('nps_mean')
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cat_colors = ['#e63946' if n < 5 else '#06d6a0' for n in cat_nps['nps_mean']]
axes[0].barh(cat_nps.index, cat_nps['nps_mean'], color=cat_colors)
axes[0].axvline(5, color='gray', linestyle='--', linewidth=1)
axes[0].set_xlabel('NPS medio')
axes[0].set_title('NPS medio por categoría predicha')

axes[1].barh(cat_nps.index, cat_nps['pct_urgente']*100, color='#e63946')
axes[1].set_xlabel('% urgentes')
axes[1].set_title('% Tickets urgentes por categoría')

plt.tight_layout()
plt.show()

print('\nTop 3 categorías con mayor % urgente:')
print(cat_nps.sort_values('pct_urgente', ascending=False)[['n','nps_mean','pct_urgente']].head(3))

## 7. Motor de búsqueda: encontrar tickets similares al query

In [ ]:
# Indexar corpus completo
tfidf_search = TfidfVectorizer(preprocessor=preprocess, stop_words=STOPWORDS_ES, ngram_range=(1,2))
X_index = tfidf_search.fit_transform(df['comment'])

def search_tickets(query, top_k=5):
    q = tfidf_search.transform([query])
    sims = cosine_similarity(q, X_index).ravel()
    top  = np.argsort(sims)[::-1][:top_k]
    print(f'\nQuery: "{query}"')
    print(f'{"Ticket":<55} {"Sim":>5}  {"Cat":>18}  {"NPS":>4}')
    print('-'*88)
    for i in top:
        print(f'{df.comment.iloc[i]:<55} {sims[i]:5.3f}  '
              f'{df.pred_category.iloc[i]:>18}  {df.nps_score.iloc[i]:>4}')

search_tickets('la aplicación no funciona y no carga')
search_tickets('nunca llega el código SMS')

## 8. Resumen ejecutivo

In [ ]:
nps_promoters  = (df.nps_score >= 9).sum()
nps_detractors = (df.nps_score <= 6).sum()
nps_net = (nps_promoters - nps_detractors) / len(df) * 100

worst_cat = cat_nps.sort_values('nps_mean').index[0]
worst_ch  = channel_summary.sort_values('nps_mean').index[0]
best_ch   = channel_summary.sort_values('nps_mean').index[-1]

print('=' * 60)
print('RESUMEN EJECUTIVO — VOZ DEL CLIENTE')
print('=' * 60)
print(f'Encuestas analizadas:   {len(df):>8,}')
print(f'NPS score (net):        {nps_net:>+7.1f}%')
print(f'  Promotores (9-10):    {nps_promoters:>8,}  ({nps_promoters/len(df):.0%})')
print(f'  Detractores (0-6):    {nps_detractors:>8,}  ({nps_detractors/len(df):.0%})')
print()
print(f'Tickets urgentes:       {df.urgent.sum():>8,}  ({df.urgent.mean():.0%})')
print(f'Categoría más crítica:  {worst_cat}')
print(f'Canal más satisfecho:   {best_ch}')
print(f'Canal más problemático: {worst_ch}')
print()
print('Top 3 temas por volumen:')
for cat, n in df.pred_category.value_counts().head(3).items():
    print(f'  {cat:<25} {n:>4} comentarios')
print('=' * 60)

## Lo que usamos en este notebook
| Técnica | Aplicada como |
|---|---|
| Preprocesamiento | `preprocess()` — regex + lowercase |
| TF-IDF | Representación vectorial para clasificación y búsqueda |
| Clasificación multiclase | `LogisticRegression(multi_class='multinomial')` |
| Clasificación binaria | Sentimiento positivo/negativo |
| Reglas de negocio | `urgent = negativo AND nps ≤ 4` |
| Búsqueda semántica | `cosine_similarity(query, corpus)` |
| Análisis por segmento | `groupby('channel').agg(...)` con predicciones |
| KPIs de NLP | NPS net score, % urgentes, categoría dominante por canal |

---
**Curso completo.** Tienes todas las herramientas para construir un sistema NLP de producción usando solo sklearn, numpy y pandas — sin dependencias externas.